# Notebook 04 — BindCraft: Designing Binders Against PD-L1

**Learning objectives:**
- Set up and run the BindCraft protein design pipeline
- Configure the target (PD-L1) and hotspot residues
- Monitor the design loop in real time
- Visualize the top accepted design

**Time:** 15 min setup + 1–3 hours running + 30 min analysis

**Companion wiki pages:** [6.1 Setup](https://rucha1796.github.io/internship-bindcraft-2026/m6_01_setup/) | [6.2 Running](https://rucha1796.github.io/internship-bindcraft-2026/m6_02_running/) | [5.2 Pipeline](https://rucha1796.github.io/internship-bindcraft-2026/m5_02_bindcraft_pipeline/)

---
> **GPU required.** Runtime → Change runtime type → T4 GPU  
> **Google Drive recommended.** Mount Drive to save outputs persistently (Cell 1).

---
> **No GPU available?** Skip to Cell 10 — the pre-generated dataset is automatically loaded and you can do all the analysis.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory on Drive
import os
WORK_DIR = "/content/drive/MyDrive/bindcraft"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"✓ Drive mounted. Working directory: {WORK_DIR}")

## Cell 2 — Install BindCraft + dependencies

Takes ~5 minutes on first run. Subsequent runs on the same Colab session skip most downloads.

In [ ]:
%%bash
set -e

# BindCraft repository
if [ ! -d "/content/bindcraft" ]; then
    git clone --depth 1 https://github.com/martinpacesa/BindCraft.git /content/bindcraft
    echo "✓ BindCraft cloned"
fi

# ColabDesign (AF2 hallucination library)
pip install -q "colabdesign @ git+https://github.com/sokrypton/ColabDesign"
echo "✓ ColabDesign installed"

# PyRosetta (for energy scoring and relaxation)
pip install -q pyrosetta-installer
python -c "import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()" 2>&1 | tail -3
echo "✓ PyRosetta installed"

# AlphaFold2 parameters
cd /content/bindcraft && bash install_bindcraft.sh --hardware=GPU 2>&1 | tail -5
echo "✓ BindCraft setup complete"

## Cell 3 — Settings

Configure the target protein and design parameters here.

**Do not change `target_hotspot_residues` without reading Module 3.3 first.**

In [ ]:
import json, os

# ============================================================
#  SETTINGS — Edit these values
# ============================================================

design_path         = "/content/drive/MyDrive/bindcraft/"  # Where outputs are saved
binder_name         = "PDL1"                                # Label for this run
starting_pdb        = "/content/bindcraft/example/PDL1.pdb" # Target structure
chains              = "A"                                   # Which chain is the target
target_hotspot_residues = "54,56,111,115,123"               # PD-L1 hotspot residues
length_min          = 65                                    # Min binder length (residues)
length_max          = 90                                    # Max binder length (residues)
number_of_final_designs = 10                                # Stop when this many pass all filters

# ============================================================

# Write settings to JSON
settings = {
    "binder_name": binder_name,
    "starting_pdb": starting_pdb,
    "chains": chains,
    "target_hotspot_residues": target_hotspot_residues,
    "lengths": [length_min, length_max],
    "number_of_final_designs": number_of_final_designs
}

settings_path = f"{design_path}/{binder_name}.json"
os.makedirs(design_path, exist_ok=True)
with open(settings_path, "w") as f:
    json.dump(settings, f, indent=4)

print("=== BindCraft Settings ===")
for k, v in settings.items():
    print(f"  {k:35s}: {v}")
print(f"\nSaved to: {settings_path}")

## Cell 4 — Advanced settings (defaults are fine)

These control the hallucination protocol. The defaults work well for helical binders against globular targets.

In [ ]:
# Advanced settings — leave as defaults for first run
advanced_settings = {
    "design_protocol": "default",     # Hallucination loss function
    "prediction_protocol": "default", # 5 AF2 models per design
    "interface_protocol": "default",  # Interface scoring
    "template_protocol": "default"    # No template (true de novo)
}

print("Advanced settings (defaults):")
for k, v in advanced_settings.items():
    print(f"  {k}: {v}")
print("\nNote: These can be modified for advanced use cases (see wiki Module 6.1)")

## Cell 5 — Filter preset

Choose how strict the quality filters are:
- **Default** — recommended for first run
- **Relaxed** — slightly lower thresholds; use if acceptance rate is < 5%
- **None** — no filtering; not recommended (produces low-quality designs)

In [ ]:
filter_preset = "default"  # Options: "default", "relaxed", "peptide", "none"

filter_thresholds = {
    "default": {
        "i_pTM": 0.55, "pLDDT": 0.85, "Binder_pLDDT": 0.80,
        "i_pAE": 10, "ShapeComplementarity": 0.55,
        "dG": -10, "Clashes": 5, "Binder_RMSD": 2.0
    },
    "relaxed": {
        "i_pTM": 0.45, "pLDDT": 0.80, "Binder_pLDDT": 0.70,
        "i_pAE": 15, "ShapeComplementarity": 0.45,
        "dG": -5, "Clashes": 10, "Binder_RMSD": 3.0
    }
}

thresholds = filter_thresholds.get(filter_preset, filter_thresholds["default"])
print(f"Filter preset: {filter_preset}")
print("\nThresholds:")
for k, v in thresholds.items():
    print(f"  {k}: {v}")

## Cell 6 — Import BindCraft functions

In [ ]:
import sys
sys.path.insert(0, "/content/bindcraft")

from bindcraft import BindCraft
import os

print("✓ BindCraft imported")
print(f"  Target PDB: {starting_pdb}")
print(f"  PDB exists: {os.path.exists(starting_pdb)}")

if not os.path.exists(starting_pdb):
    print("\n⚠ PDB not found at that path. Check Cell 3 settings.")
    print("  The example PDL1.pdb should be at /content/bindcraft/example/PDL1.pdb")
    print("  Run: !ls /content/bindcraft/example/ to check available files")

## Cell 7 — Initialize PyRosetta

In [ ]:
import pyrosetta
pyrosetta.init("-mute all")
print("✓ PyRosetta initialized")

## Cell 8 — Run BindCraft

This is the main design loop. It will run until it collects `number_of_final_designs` (10) accepted designs.

**Expected time:** 1–3 hours depending on GPU and acceptance rate.

**What you'll see printed:**
```
Trajectory 1: i_pTM=0.23 [REJECTED - low i_pTM]
Trajectory 2: i_pTM=0.61 → MPNN → ACCEPTED (1/10)
...
```

**Do not close this browser tab while running.**

In [ ]:
# Main BindCraft design loop
bc = BindCraft(
    settings_path=settings_path,
    design_path=design_path,
    advanced_settings=advanced_settings,
    filters=thresholds
)

bc.run()

print("\n" + "="*50)
print("BindCraft run complete!")
print(f"Results saved to: {design_path}/{binder_name}/")
print("="*50)

## Cell 9 — Consolidate and rank results

In [ ]:
import pandas as pd
import os

results_dir = f"{design_path}/{binder_name}"
final_csv = f"{results_dir}/final_design_stats.csv"

if os.path.exists(final_csv):
    df = pd.read_csv(final_csv)
    df_ranked = df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
    
    print(f"Total accepted designs: {len(df)}")
    print(f"\nTop 10 designs (ranked by i_pTM):")
    
    key_cols = [
        "binder_name", "Average_i_pTM", "Average_pLDDT",
        "Average_Binder_pLDDT", "Average_Binder_RMSD",
        "Average_ShapeComplementarity", "Average_dG"
    ]
    print(df_ranked.head(10)[key_cols].to_string(index=True))
else:
    print(f"CSV not found at {final_csv}")
    print("Either the run is still in progress, or use the pregenerated dataset (Cell 10)")

---
## Cell 10 — Load pregenerated dataset (use this if no GPU or run incomplete)

The pregenerated `PDL1_8aok` dataset has 12 accepted designs from a full BindCraft run. All analysis notebooks use this as the default data source.

In [ ]:
import pandas as pd
import os

# ============================================================
# SWITCH: Set to True to use pregenerated data, False to use your own run
USE_PREGENERATED = True
# ============================================================

if USE_PREGENERATED:
    # Pregenerated dataset (should be on Drive from internship setup)
    PREGEN_PATH = "/content/drive/MyDrive/bindcraft/PDL1_8aok"
    
    # Fallback: generate a sample dataset for demonstration
    if not os.path.exists(f"{PREGEN_PATH}/final_design_stats.csv"):
        print("Pregenerated dataset not found on Drive.")
        print("Ask your mentor for the PDL1_8aok dataset, or run Cell 8 to generate your own.")
        print("\nCreating a minimal example dataset for demonstration...")
        
        import numpy as np
        np.random.seed(42)
        n = 12
        demo_data = {
            "binder_name": [f"PDL1_traj{i}_mpnn1" for i in range(1, n+1)],
            "binder_sequence": ["MKVIFGLMVGGVIA" * 5 for _ in range(n)],
            "Average_i_pTM": np.random.uniform(0.56, 0.74, n),
            "Average_pLDDT": np.random.uniform(0.86, 0.94, n),
            "Average_Binder_pLDDT": np.random.uniform(0.81, 0.91, n),
            "Average_Binder_RMSD": np.random.uniform(0.5, 1.8, n),
            "Average_ShapeComplementarity": np.random.uniform(0.56, 0.72, n),
            "Average_dG": np.random.uniform(-22, -11, n),
            "Average_i_pAE": np.random.uniform(5, 9, n),
            "Average_n_InterfaceHbonds": np.random.randint(4, 12, n).astype(float),
            "Average_Relaxed_Clashes": np.random.uniform(0, 4, n),
        }
        df = pd.DataFrame(demo_data)
        df = df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
        os.makedirs(PREGEN_PATH, exist_ok=True)
        df.to_csv(f"{PREGEN_PATH}/final_design_stats.csv", index=False)
        print(f"  Created demo dataset with {n} designs.")
    
    df = pd.read_csv(f"{PREGEN_PATH}/final_design_stats.csv")
    results_dir = PREGEN_PATH
else:
    results_dir = f"{design_path}/{binder_name}"
    df = pd.read_csv(f"{results_dir}/final_design_stats.csv")

print(f"Dataset loaded from: {results_dir}")
print(f"Designs available: {len(df)}")
df_ranked = df.sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)

## Cell 11 — View top 20 designs

In [ ]:
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)

key_cols = [
    "binder_name",
    "Average_i_pTM",
    "Average_pLDDT",
    "Average_Binder_pLDDT",
    "Average_Binder_RMSD",
    "Average_ShapeComplementarity",
    "Average_dG",
    "Average_n_InterfaceHbonds",
    "Average_Relaxed_Clashes"
]

n_show = min(20, len(df_ranked))
print(f"Top {n_show} designs by i_pTM:")
print("="*120)
print(df_ranked.head(n_show)[key_cols].to_string())

best = df_ranked.iloc[0]
print(f"\n→ Best design: {best['binder_name']}")
print(f"  i_pTM = {best['Average_i_pTM']:.3f}")
print(f"  ShapeComplementarity = {best['Average_ShapeComplementarity']:.3f}")
print(f"  Binder RMSD = {best['Average_Binder_RMSD']:.2f} Å")
print(f"  Sequence: {best.get('binder_sequence', 'N/A')[:50]}...")

## Cell 12 — Visualize the top design

Load and display the best accepted design in 3D.

In [ ]:
import py3Dmol
import glob

# Find PDB files for the top design
ranked_pdbs = sorted(glob.glob(f"{results_dir}/Accepted/Ranked/*.pdb"))

if ranked_pdbs:
    top_pdb_path = ranked_pdbs[0]  # Rank 1 = best
    print(f"Loading: {top_pdb_path}")
    
    with open(top_pdb_path) as f:
        pdb_str = f.read()
    
    view = py3Dmol.view(width=800, height=550)
    view.addModel(pdb_str, "pdb")
    
    # Chain A = PD-L1 target (steel blue)
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})
    
    # Chain B = designed binder (rose gold)
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "#B76E79"}})
    
    # Semi-transparent surface on PD-L1
    view.addSurface(py3Dmol.VDW, {"opacity": 0.3, "color": "#3c5b6f"}, {"chain": "A"})
    
    # Hotspot residues as yellow spheres
    for resnum in [54, 56, 111, 115, 123]:
        view.addStyle(
            {"chain": "A", "resi": str(resnum)},
            {"sphere": {"color": "yellow", "radius": 0.9}}
        )
    
    view.zoomTo()
    view.setBackgroundColor("white")
    view.show()
    
    print(f"\nBlue = PD-L1 target | Rose = designed binder | Yellow = hotspot residues")
    print(f"\nBest design metrics:")
    print(f"  i_pTM:               {best['Average_i_pTM']:.3f}")
    print(f"  ShapeComplementarity: {best['Average_ShapeComplementarity']:.3f}")
    print(f"  Binder RMSD:          {best['Average_Binder_RMSD']:.2f} Å")
    print(f"  dG:                   {best['Average_dG']:.1f} kcal/mol")

else:
    print("No PDB files found in Accepted/Ranked/")
    print("The pregenerated dataset may not include PDB files.")
    print("Run Cell 8 to generate your own, or ask your mentor for the full pregenerated dataset.")

---
## Summary

In this notebook you:
- Set up BindCraft with the PD-L1 target and 5 hotspot residues
- Ran (or loaded) the design pipeline
- Viewed the top 10–20 accepted designs ranked by i_pTM
- Visualized the best design in 3D: PD-L1 (blue) + designed binder (rose)

**Next:** Notebook 05 — Deep analysis of the BindCraft CSV outputs with pandas